<a href="https://colab.research.google.com/github/uniqueeee777/stability-aware-multiview-food-security/blob/main/04_Extended_GridSearch_R0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Load data dan kunci parameter Grid Search

In [ ]:

# ============================================================
# CELL 1 — EXTENDED GRID SEARCH: PROTOCOL & DATA
# ============================================================

!pip -q install "kmedoids==0.5.5"

from google.colab import drive
from pathlib import Path
from datetime import datetime, timezone
from importlib.metadata import version

from scipy.spatial.distance import pdist, squareform
from scipy.cluster.hierarchy import linkage, cut_tree
from scipy.sparse.csgraph import connected_components
from sklearn.cluster import SpectralClustering
from sklearn.metrics import silhouette_score, silhouette_samples
from IPython.display import display

import numpy as np
import pandas as pd
import scipy
import sklearn
import kmedoids
import hashlib
import json

drive.mount("/content/drive")

ROOT = Path(
    "/content/drive/MyDrive/"
    "Research_Climate_FoodSecurity_85Kab"
)

SOURCE = (
    ROOT / "01_data" / "01_raw"
    / "MATRIX_9VAR_CLUSTERING_85KAB_2025.csv"
)

REP_DIR = (
    ROOT / "03_outputs"
    / "02_representation"
    / "20261018_173812_UTC"
)

EXPECTED_HASH = (
    "ce932d322eb49cbdb6ea3b0a8768ca89c2e9"
    "e7f0722232883541d891089ca6fa"
)

assert SOURCE.exists()
assert hashlib.sha256(
    SOURCE.read_bytes()
).hexdigest() == EXPECTED_HASH

df = pd.read_csv(
    SOURCE,
    dtype={"kode_bps": "string"}
)

with np.load(
    REP_DIR / "R0_raw_zscore_distances.npz",
    allow_pickle=True
) as f:

    D_ref = f["fused"].copy()
    Phi = f["euclidean_features"].copy()

    ids = f["kode_bps"].astype(str)

    D_views = {
        "Food": f["food"].copy(),
        "Climate": f["climate"].copy(),
        "Accessibility": f["accessibility"].copy()
    }

assert np.array_equal(
    ids,
    df["kode_bps"].astype(str).to_numpy()
)

assert D_ref.shape == (85, 85)
assert np.allclose(D_ref, D_ref.T)
assert np.allclose(np.diag(D_ref), 0)
assert np.isfinite(D_ref).all()

assert np.allclose(
    squareform(pdist(Phi)),
    D_ref,
    atol=1e-10
)

# ------------------------------------------------------------
# GRID PROTOCOL
# ------------------------------------------------------------

SEED = 20261018

K_GRID = [2, 3, 4, 5, 6]
K_GRAPH_GRID = [5, 8, 12, 16, 24]
K_SCALE_GRID = [5, 7, 10]

VIEW_NAMES = ["Food", "Climate", "Accessibility"]

WEIGHT_GRID = {
    "equal": (1/3, 1/3, 1/3),
    "food50": (0.50, 0.25, 0.25),
    "climate50": (0.25, 0.50, 0.25),
    "access50": (0.25, 0.25, 0.50)
}

for weights in WEIGHT_GRID.values():
    assert np.isclose(sum(weights), 1)
    assert all(w > 0 for w in weights)

plan = []

# PAM + Hierarchical: 4 methods × 5 K = 20
for method in [
    "KMedoids_PAM",
    "Hierarchical_Ward",
    "Hierarchical_Average",
    "Hierarchical_Complete"
]:
    for K in K_GRID:
        plan.append({
            "model_id": f"{method}_K{K}",
            "method": method,
            "K": K,
            "k_graph": np.nan,
            "k_scale": np.nan,
            "weight_id": "na"
        })

# Spectral Early: 5 K × 5 graph-k × 3 scale-k
for kg in K_GRAPH_GRID:
    for ks in K_SCALE_GRID:
        for K in K_GRID:
            plan.append({
                "model_id": f"Early_K{K}_G{kg}_S{ks}",
                "method": "Spectral_Early",
                "K": K,
                "k_graph": kg,
                "k_scale": ks,
                "weight_id": "na"
            })

# Spectral Multi-View
for weight_id in WEIGHT_GRID:
    for kg in K_GRAPH_GRID:
        for ks in K_SCALE_GRID:
            for K in K_GRID:
                plan.append({
                    "model_id": (
                        f"Multi_K{K}_G{kg}_S{ks}_{weight_id}"
                    ),
                    "method": "Spectral_MultiView",
                    "K": K,
                    "k_graph": kg,
                    "k_scale": ks,
                    "weight_id": weight_id
                })

grid_plan = pd.DataFrame(plan)

assert len(grid_plan) == 395
assert grid_plan["model_id"].is_unique

RUN_ID = datetime.now(timezone.utc).strftime(
    "%Y%m%d_%H%M%S_UTC"
)

OUT4 = (
    ROOT / "03_outputs"
    / "05_extended_gridsearch_R0" / RUN_ID
)
OUT4.mkdir(parents=True, exist_ok=True)

grid_plan.to_csv(
    OUT4 / "grid_plan.csv",
    index=False
)

protocol = {
    "stage": "Extended quality grid search R0",
    "run_id": RUN_ID,
    "input_sha256": EXPECTED_HASH,
    "representation": "R0_raw_zscore",
    "reference_distance": "R0_fused_euclidean",
    "K_grid": K_GRID,
    "k_graph_grid": K_GRAPH_GRID,
    "k_scale_grid": K_SCALE_GRID,
    "weight_grid": WEIGHT_GRID,
    "random_seed": SEED,
    "pam_init": "build",
    "spectral_assign_labels": "kmeans",
    "spectral_n_init": 20,
    "graph_type": "symmetric_union_kNN",
    "affinity": "local_scaling_gaussian",
    "fusion_normalization": "frobenius",
    "status": "QUALITY_GRID_NOT_FINAL_SELECTION",
    "versions": {
        "python_kmedoids": version("kmedoids"),
        "scikit_learn": sklearn.__version__,
        "scipy": scipy.__version__,
        "numpy": np.__version__,
        "pandas": pd.__version__
    }
}

(OUT4 / "protocol.json").write_text(
    json.dumps(protocol, indent=2),
    encoding="utf-8"
)

print("EXTENDED GRID PROTOCOL LOCKED")
print("Candidates:", len(grid_plan))
print("Dataset:", D_ref.shape)
print("Output:", OUT4)

display(grid_plan.groupby("method").size())


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
EXTENDED GRID PROTOCOL LOCKED
Candidates: 395
Dataset: (85, 85)
Output: /content/drive/MyDrive/Research_Climate_FoodSecurity_85Kab/03_outputs/05_extended_gridsearch_R0/20261019_014117_UTC


,0
method,
Hierarchical_Average,5
Hierarchical_Complete,5
Hierarchical_Ward,5
KMedoids_PAM,5
Spectral_Early,75
Spectral_MultiView,300


# Grid Search dan quality diagnostics

In [ ]:

# ============================================================
# CELL 2 — EXTENDED GRID SEARCH EXECUTION
# No stability-based selection in this stage
# ============================================================

# ------------------------------------------------------------
# 1. REPRODUCIBLE GRAPH CONSTRUCTION
# ------------------------------------------------------------

def build_graph(D, kg, ks):

    D = np.asarray(D, dtype=float)
    n = D.shape[0]

    if D.shape != (n, n):
        raise ValueError("Distance matrix not square.")

    if not np.allclose(D, D.T):
        raise ValueError("Distance matrix not symmetric.")

    if not np.allclose(np.diag(D), 0):
        raise ValueError("Nonzero distance diagonal.")

    if not np.isfinite(D).all() or (D < 0).any():
        raise ValueError("Invalid distance values.")

    if not (1 <= kg < n and 1 <= ks < n):
        raise ValueError("Invalid graph parameters.")

    A = D.copy()
    np.fill_diagonal(A, np.inf)

    order = np.argsort(
        A, axis=1, kind="stable"
    )

    sigma = np.take_along_axis(
        A,
        order[:, [ks - 1]],
        axis=1
    ).ravel()

    if (sigma <= 0).any() or not np.isfinite(sigma).all():
        raise ValueError("Invalid local scales.")

    adjacency = np.zeros((n, n), dtype=bool)

    adjacency[
        np.arange(n)[:, None],
        order[:, :kg]
    ] = True

    adjacency |= adjacency.T

    W = np.exp(
        -(D ** 2) /
        (sigma[:, None] * sigma[None, :])
    )

    W = np.where(adjacency, W, 0.0)
    np.fill_diagonal(W, 0.0)

    if not np.allclose(W, W.T):
        raise ValueError("Graph not symmetric.")

    components = connected_components(
        (W > 0).astype(np.uint8),
        directed=False
    )[0]

    if components != 1:
        raise ValueError(
            f"Disconnected graph: {components} components."
        )

    return W


EARLY_CACHE = {}
VIEW_CACHE = {}

def get_affinity(method, kg, ks, weight_id):

    key = (kg, ks)

    if method == "Spectral_Early":

        if key not in EARLY_CACHE:
            EARLY_CACHE[key] = build_graph(
                D_ref, kg, ks
            )

        return EARLY_CACHE[key]

    if method == "Spectral_MultiView":

        if key not in VIEW_CACHE:

            normalized = {}

            for view, D in D_views.items():
                Wv = build_graph(D, kg, ks)

                norm = np.linalg.norm(
                    Wv, ord="fro"
                )

                if norm <= 0 or not np.isfinite(norm):
                    raise ValueError("Invalid Frobenius norm.")

                normalized[view] = Wv / norm

            VIEW_CACHE[key] = normalized

        weights = WEIGHT_GRID[weight_id]

        W = sum(
            weights[i] * VIEW_CACHE[key][view]
            for i, view in enumerate(VIEW_NAMES)
        )

        return W

    raise ValueError(f"Unknown spectral method: {method}")


# ------------------------------------------------------------
# 2. PRECOMPUTE HIERARCHICAL TREES
# ------------------------------------------------------------

condensed = squareform(
    D_ref, checks=True
)

TREES = {
    "Hierarchical_Ward": linkage(
        Phi, method="ward", metric="euclidean"
    ),
    "Hierarchical_Average": linkage(
        condensed, method="average"
    ),
    "Hierarchical_Complete": linkage(
        condensed, method="complete"
    )
}


# ------------------------------------------------------------
# 3. FIT ONE CONFIGURATION
# ------------------------------------------------------------

def fit_grid_model(row):

    method = row["method"]
    K = int(row["K"])

    if method == "KMedoids_PAM":

        fitted = kmedoids.pam(
            D_ref,
            K,
            init="build",
            max_iter=300,
            random_state=SEED
        )

        labels = fitted.labels
        graph_density = np.nan

    elif method in TREES:

        labels = cut_tree(
            TREES[method],
            n_clusters=K
        ).ravel()

        graph_density = np.nan

    elif method.startswith("Spectral_"):

        kg = int(row["k_graph"])
        ks = int(row["k_scale"])

        W = get_affinity(
            method, kg, ks, row["weight_id"]
        )

        model = SpectralClustering(
            n_clusters=K,
            affinity="precomputed",
            assign_labels="kmeans",
            eigen_solver="arpack",
            n_init=20,
            random_state=SEED
        )

        labels = model.fit_predict(W)

        graph_density = (
            np.count_nonzero(np.triu(W, k=1))
            / (len(W) * (len(W) - 1) / 2)
        )

    else:
        raise ValueError(f"Unknown method: {method}")

    labels = np.asarray(labels, dtype=int)

    if len(labels) != len(df):
        raise ValueError("Invalid label dimensions.")

    unique, sizes = np.unique(
        labels, return_counts=True
    )

    if len(unique) != K:
        raise ValueError(
            f"Expected {K} clusters, got {len(unique)}."
        )

    return labels, sizes, graph_density


# ------------------------------------------------------------
# 4. EVALUATION ON COMMON REFERENCE DISTANCE
# ------------------------------------------------------------

def quality_metrics(labels, sizes):

    sample_sil = silhouette_samples(
        D_ref, labels, metric="precomputed"
    )

    scores = {
        "silhouette_R0": float(sample_sil.mean()),
        "negative_silhouette_pct": float(
            100 * (sample_sil < 0).mean()
        ),
        "smallest_cluster": int(sizes.min()),
        "largest_cluster": int(sizes.max()),
        "n_clusters_size_le_2": int((sizes <= 2).sum()),
        "cluster_sizes": json.dumps(sizes.tolist())
    }

    for view, D in D_views.items():

        scores[f"sil_{view.lower()}"] = float(
            silhouette_score(
                D, labels,
                metric="precomputed"
            )
        )

    return scores


# ------------------------------------------------------------
# 5. EXECUTE THE COMPLETE GRID
# ------------------------------------------------------------

results_list = []
failure_list = []
label_store = {}

for count, row in enumerate(
    grid_plan.to_dict("records"),
    start=1
):

    model_id = row["model_id"]

    try:

        labels, sizes, density = fit_grid_model(row)

        metrics = quality_metrics(labels, sizes)

        results_list.append({
            **row,
            **metrics,
            "graph_density": density,
            "status": "SUCCESS"
        })

        label_store[model_id] = labels

    except Exception as exc:

        failure_list.append({
            **row,
            "error_type": type(exc).__name__,
            "error_message": str(exc)
        })

    # Checkpoint without changing current results
    if count % 25 == 0:

        pd.DataFrame(results_list).to_csv(
            OUT4 / "quality_checkpoint.csv",
            index=False
        )

        pd.DataFrame(failure_list).to_csv(
            OUT4 / "failures_checkpoint.csv",
            index=False
        )

        print(
            f"Evaluated {count}/{len(grid_plan)} "
            f"| Success: {len(results_list)} "
            f"| Failed: {len(failure_list)}"
        )


# ------------------------------------------------------------
# 6. SAVE FINAL GRID OUTPUTS
# ------------------------------------------------------------

quality_results = pd.DataFrame(results_list)
failures = pd.DataFrame(
    failure_list,
    columns=list(grid_plan.columns) + [
        "error_type", "error_message"
    ]
)

labels_output = pd.DataFrame({
    "kode_bps": df["kode_bps"].astype(str)
})

if label_store:
    labels_output = pd.concat(
        [
            labels_output,
            pd.DataFrame(label_store)
        ],
        axis=1
    )

assert len(quality_results) + len(failures) == 395
assert quality_results["model_id"].is_unique

quality_results.to_csv(
    OUT4 / "quality_grid_all.csv",
    index=False
)

failures.to_csv(
    OUT4 / "failed_configurations.csv",
    index=False
)

labels_output.to_csv(
    OUT4 / "labels_all.csv",
    index=False
)

print("\nEXTENDED GRID SEARCH COMPLETED")
print("Successful:", len(quality_results))
print("Failed:", len(failures))

if len(failures):
    print("\nFAILED CONFIGURATIONS REQUIRE REVIEW:")
    display(failures)
else:
    print("No fitting failures.")

print("\nSaved:", OUT4)


Evaluated 25/395 | Success: 25 | Failed: 0
Evaluated 50/395 | Success: 50 | Failed: 0
Evaluated 75/395 | Success: 75 | Failed: 0
Evaluated 100/395 | Success: 100 | Failed: 0
Evaluated 125/395 | Success: 125 | Failed: 0
Evaluated 150/395 | Success: 150 | Failed: 0
Evaluated 175/395 | Success: 175 | Failed: 0
Evaluated 200/395 | Success: 200 | Failed: 0
Evaluated 225/395 | Success: 225 | Failed: 0
Evaluated 250/395 | Success: 250 | Failed: 0
Evaluated 275/395 | Success: 275 | Failed: 0
Evaluated 300/395 | Success: 300 | Failed: 0
Evaluated 325/395 | Success: 325 | Failed: 0
Evaluated 350/395 | Success: 350 | Failed: 0
Evaluated 375/395 | Success: 375 | Failed: 0

EXTENDED GRID SEARCH COMPLETED
Successful: 395
Failed: 0
No fitting failures.

Saved: /content/drive/MyDrive/Research_Climate_FoodSecurity_85Kab/03_outputs/05_extended_gridsearch_R0/20261019_014117_UTC


# Audit dan ringkasan Grid Search

In [ ]:

# ============================================================
# CELL 3 — QUALITY GRID AUDIT
# Diagnostic summary, not final selection
# ============================================================

from sklearn.metrics import adjusted_rand_score

assert len(quality_results) + len(failures) == 395

print("A. GRID COVERAGE")

coverage = (
    quality_results.groupby("method")
    .agg(
        n_success=("model_id", "count"),
        max_silhouette=("silhouette_R0", "max"),
        min_silhouette=("silhouette_R0", "min")
    )
    .reset_index()
)

display(coverage.round(4))

print("\nB. BEST OBSERVED CONFIGURATION PER METHOD")

best_indices = (
    quality_results.groupby("method")["silhouette_R0"]
    .idxmax()
)

best_per_method = (
    quality_results.loc[
        best_indices,
        [
            "model_id", "method", "K",
            "k_graph", "k_scale", "weight_id",
            "silhouette_R0",
            "sil_food", "sil_climate",
            "sil_accessibility",
            "smallest_cluster",
            "graph_density"
        ]
    ]
    .sort_values(
        "silhouette_R0", ascending=False
    )
)

display(best_per_method.round(4))

print("\nC. HIGHEST SILHOUETTE CONFIGURATIONS")

display(
    quality_results.sort_values(
        "silhouette_R0",
        ascending=False
    )[
        [
            "model_id", "K",
            "silhouette_R0",
            "smallest_cluster",
            "graph_density"
        ]
    ].head(20).round(4)
)

# ------------------------------------------------------------
# D. CROSS-CHECK OLD PILOT CONFIGURATIONS
# ------------------------------------------------------------

old_dir = (
    ROOT / "03_outputs"
    / "03_gridsearch_pilot"
    / "20261018_175911_UTC"
)

old = pd.read_csv(
    old_dir / "gridsearch_R0_all_candidates.csv"
)

# Equivalent combinations in the new grid:
# traditional (no graph parameters),
# spectral k_scale=7 and equal fusion weights.

compatible = quality_results.loc[
    (
        quality_results["method"].isin([
            "KMedoids_PAM",
            "Hierarchical_Ward",
            "Hierarchical_Average",
            "Hierarchical_Complete"
        ])
    )
    |
    (
        (quality_results["method"] == "Spectral_Early")
        & (quality_results["k_scale"] == 7)
    )
    |
    (
        (quality_results["method"] == "Spectral_MultiView")
        & (quality_results["k_scale"] == 7)
        & (quality_results["weight_id"] == "equal")
    )
].copy()

assert len(compatible) <= 70

def join_key(row):
    kg = (
        -1
        if pd.isna(row["k_graph"])
        else int(row["k_graph"])
    )
    return f"{row['method']}|{int(row['K'])}|{kg}"

old["join_key"] = old.apply(join_key, axis=1)
compatible["join_key"] = compatible.apply(join_key, axis=1)

comparison = old[
    ["join_key", "silhouette"]
].merge(
    compatible[
        ["join_key", "silhouette_R0"]
    ],
    on="join_key",
    how="left",
    validate="one_to_one"
)

comparison["abs_difference"] = (
    comparison["silhouette"]
    - comparison["silhouette_R0"]
).abs()

print("\nD. REPRODUCIBILITY CHECK AGAINST 70 PILOT MODELS")
print("Old configurations:", len(comparison))
print(
    "Successfully matched:",
    int(comparison["silhouette_R0"].notna().sum())
)
print(
    "Maximum score difference:",
    comparison["abs_difference"].max()
)

mismatches = comparison.loc[
    comparison["silhouette_R0"].isna()
    | (comparison["abs_difference"] > 1e-6)
]

if mismatches.empty:
    print("Pilot quality results reproduced.")
else:
    print("Review the following mismatches:")
    display(mismatches)

comparison.to_csv(
    OUT4 / "pilot_reproducibility_check.csv",
    index=False
)

best_per_method.to_csv(
    OUT4 / "quality_best_per_method.csv",
    index=False
)

print("\nCELL 3 COMPLETED")


A. GRID COVERAGE


,method,n_success,max_silhouette,min_silhouette
0,Hierarchical_Average,5,0.3203,0.2184
1,Hierarchical_Complete,5,0.3090,0.1647
2,Hierarchical_Ward,5,0.2805,0.1998
3,KMedoids_PAM,5,0.2835,0.2068
4,Spectral_Early,75,0.2904,0.1637
5,Spectral_MultiView,300,0.3127,0.0098



B. BEST OBSERVED CONFIGURATION PER METHOD


,model_id,method,K,k_graph,k_scale,weight_id,silhouette_R0,sil_food,sil_climate,sil_accessibility,smallest_cluster,graph_density
10,Hierarchical_Average_K2,Hierarchical_Average,2,NaN,NaN,na,0.3203,0.1497,0.2732,0.4792,4,NaN
175,Multi_K2_G5_S7_food50,Spectral_MultiView,2,5.0,7.0,food50,0.3127,0.4703,-0.0043,0.3774,19,0.2022
15,Hierarchical_Complete_K2,Hierarchical_Complete,2,NaN,NaN,na,0.3090,0.4130,-0.0364,0.4303,14,NaN
71,Early_K3_G16_S7,Spectral_Early,3,16.0,7.0,na,0.2904,0.1536,0.2332,0.0471,17,0.2479
1,KMedoids_PAM_K3,KMedoids_PAM,3,NaN,NaN,na,0.2835,0.1477,0.2338,0.0589,15,NaN
5,Hierarchical_Ward_K2,Hierarchical_Ward,2,NaN,NaN,na,0.2805,0.4799,0.0095,0.2678,18,NaN



C. HIGHEST SILHOUETTE CONFIGURATIONS


,model_id,K,silhouette_R0,smallest_cluster,graph_density
10,Hierarchical_Average_K2,2,0.3203,4,NaN
180,Multi_K2_G5_S10_food50,2,0.3127,19,0.2022
175,Multi_K2_G5_S7_food50,2,0.3127,19,0.2022
190,Multi_K2_G8_S7_food50,2,0.3102,20,0.2966
220,Multi_K2_G16_S7_food50,2,0.3102,20,0.5246
205,Multi_K2_G12_S7_food50,2,0.3102,20,0.4171
235,Multi_K2_G24_S7_food50,2,0.3102,20,0.6975
195,Multi_K2_G8_S10_food50,2,0.3102,20,0.2966
15,Hierarchical_Complete_K2,2,0.3090,14,NaN
160,Multi_K2_G24_S7_equal,2,0.3079,21,0.6975



D. REPRODUCIBILITY CHECK AGAINST 70 PILOT MODELS
Old configurations: 70
Successfully matched: 70
Maximum score difference: 8.326672684688674e-17
Pilot quality results reproduced.

CELL 3 COMPLETED


# Partition uniqueness and Grid Search audit

In [ ]:

# ============================================================
# CELL 4 — PARTITION UNIQUENESS & GRID SEARCH AUDIT
#
# Purpose:
# Identify identical partitions despite different
# hyperparameter configurations.
#
# Diagnostic only. Do not remove candidates yet.
# ============================================================

from sklearn.metrics import adjusted_rand_score
from IPython.display import display
import numpy as np
import pandas as pd

# ------------------------------------------------------------
# 1. LOAD SAVED GRID RESULTS
# ------------------------------------------------------------

quality = pd.read_csv(
    OUT4 / "quality_grid_all.csv"
)

labels_data = pd.read_csv(
    OUT4 / "labels_all.csv",
    dtype={"kode_bps": "string"}
)

plan = pd.read_csv(
    OUT4 / "grid_plan.csv"
)

assert len(plan) == 395
assert len(quality) == 395
assert len(labels_data) == 85

assert plan["model_id"].is_unique
assert quality["model_id"].is_unique

assert set(plan["model_id"]) == set(
    quality["model_id"]
)

assert set(quality["model_id"]) == (
    set(labels_data.columns) - {"kode_bps"}
)

assert np.array_equal(
    labels_data["kode_bps"].astype(str).to_numpy(),
    df["kode_bps"].astype(str).to_numpy()
)

# ------------------------------------------------------------
# 2. CANONICAL PARTITION
#
# Different integer labels may encode the same partition.
# Canonicalization assigns labels in order of first appearance.
# Example:
# [5, 5, 9, 9] -> [0, 0, 1, 1]
# [2, 2, 0, 0] -> [0, 0, 1, 1]
# ------------------------------------------------------------

def canonical_partition(labels):

    labels = np.asarray(labels)

    canonical, _ = pd.factorize(
        labels,
        sort=False
    )

    return tuple(
        int(value)
        for value in canonical
    )


partition_lookup = {}
records = []

for row in quality.itertuples(index=False):

    model_id = row.model_id

    labels = labels_data[model_id].to_numpy(
        dtype=int
    )

    assert len(np.unique(labels)) == row.K

    key = canonical_partition(labels)

    if key not in partition_lookup:
        partition_lookup[key] = (
            f"P{len(partition_lookup) + 1:04d}"
        )

    records.append({
        "model_id": model_id,
        "partition_id": partition_lookup[key],
        "method": row.method,
        "K": int(row.K),
        "silhouette_R0": float(
            row.silhouette_R0
        ),
        "k_graph": row.k_graph,
        "k_scale": row.k_scale,
        "weight_id": row.weight_id
    })


partition_audit = pd.DataFrame(records)

# ------------------------------------------------------------
# 3. PARTITION GROUP SUMMARY
# ------------------------------------------------------------

group_summary = (
    partition_audit
    .groupby("partition_id", as_index=False)
    .agg(
        n_configurations=("model_id", "count"),
        K=("K", "first"),
        n_methods=("method", "nunique"),
        silhouette_min=("silhouette_R0", "min"),
        silhouette_max=("silhouette_R0", "max")
    )
)

group_summary["silhouette_span"] = (
    group_summary["silhouette_max"]
    - group_summary["silhouette_min"]
)

# Exactly identical partitions should have
# the same silhouette on the same distance matrix.
assert (
    group_summary["silhouette_span"] < 1e-10
).all()

n_unique = len(group_summary)
n_duplicate_excess = len(quality) - n_unique

print("A. PARTITION UNIQUENESS")
print("Grid configurations:", len(quality))
print("Unique partitions:", n_unique)
print("Repeated configurations in partition groups:",
      n_duplicate_excess)

print("\nB. UNIQUE PARTITIONS BY METHOD")

by_method = (
    partition_audit
    .groupby("method")
    .agg(
        n_configurations=("model_id", "count"),
        n_unique_partitions=("partition_id", "nunique")
    )
    .reset_index()
)

by_method["partition_repetition_pct"] = (
    100 * (
        1 - by_method["n_unique_partitions"]
        / by_method["n_configurations"]
    )
)

display(by_method.round(2))

print("\nC. MOST FREQUENTLY REPEATED PARTITIONS")

display(
    group_summary.sort_values(
        "n_configurations",
        ascending=False
    ).head(15).round(5)
)

print("\nD. CONFIGURATIONS SHARING THE SAME PARTITION")

repeated_ids = group_summary.loc[
    group_summary["n_configurations"] > 1,
    "partition_id"
]

repeated_details = partition_audit.loc[
    partition_audit["partition_id"].isin(
        repeated_ids
    )
].sort_values([
    "partition_id", "method",
    "K", "k_graph", "k_scale"
])

display(repeated_details.head(30))

# ------------------------------------------------------------
# 4. INDEPENDENT ARI CONSISTENCY CHECK
# ------------------------------------------------------------

checked_groups = 0

for pid, members in partition_audit.groupby(
    "partition_id"
):

    if len(members) < 2:
        continue

    first = members.iloc[0]["model_id"]
    second = members.iloc[1]["model_id"]

    ari = adjusted_rand_score(
        labels_data[first],
        labels_data[second]
    )

    assert np.isclose(ari, 1.0)

    checked_groups += 1

print(
    "\nGroups with independently verified ARI=1:",
    checked_groups
)

# ------------------------------------------------------------
# 5. SAVE REPRODUCIBLE OUTPUTS
# ------------------------------------------------------------

partition_audit.to_csv(
    OUT4 / "partition_uniqueness_audit.csv",
    index=False
)

group_summary.to_csv(
    OUT4 / "partition_group_summary.csv",
    index=False
)

by_method.to_csv(
    OUT4 / "partition_uniqueness_by_method.csv",
    index=False
)

print("\nPARTITION AUDIT COMPLETED")
print("Saved:", OUT4)


A. PARTITION UNIQUENESS
Grid configurations: 395
Unique partitions: 303
Repeated configurations in partition groups: 92

B. UNIQUE PARTITIONS BY METHOD


,method,n_configurations,n_unique_partitions,partition_repetition_pct
0,Hierarchical_Average,5,5,0.00
1,Hierarchical_Complete,5,5,0.00
2,Hierarchical_Ward,5,5,0.00
3,KMedoids_PAM,5,5,0.00
4,Spectral_Early,75,51,32.00
5,Spectral_MultiView,300,232,22.67



C. MOST FREQUENTLY REPEATED PARTITIONS


,partition_id,n_configurations,K,n_methods,silhouette_min,silhouette_max,silhouette_span
30,P0031,11,2,1,0.22109,0.22109,0.0
264,P0265,7,2,1,0.21338,0.21338,0.0
272,P0273,5,3,1,0.16430,0.16430,0.0
159,P0160,5,2,1,0.31017,0.31017,0.0
31,P0032,4,3,1,0.28702,0.28702,0.0
182,P0183,4,6,1,0.12752,0.12752,0.0
223,P0224,3,3,1,0.17124,0.17124,0.0
240,P0241,3,5,1,0.12802,0.12802,0.0
278,P0279,3,6,1,0.11776,0.11776,0.0
169,P0170,3,5,1,0.10657,0.10657,0.0



D. CONFIGURATIONS SHARING THE SAME PARTITION


,model_id,partition_id,method,K,silhouette_R0,k_graph,k_scale,weight_id
20,Early_K2_G5_S5,P0021,Spectral_Early,2,0.221557,5.0,5.0,na
25,Early_K2_G5_S7,P0021,Spectral_Early,2,0.221557,5.0,7.0,na
21,Early_K3_G5_S5,P0022,Spectral_Early,3,0.282886,5.0,5.0,na
26,Early_K3_G5_S7,P0022,Spectral_Early,3,0.282886,5.0,7.0,na
31,Early_K3_G5_S10,P0022,Spectral_Early,3,0.282886,5.0,10.0,na
22,Early_K4_G5_S5,P0023,Spectral_Early,4,0.201367,5.0,5.0,na
27,Early_K4_G5_S7,P0023,Spectral_Early,4,0.201367,5.0,7.0,na
32,Early_K4_G5_S10,P0023,Spectral_Early,4,0.201367,5.0,10.0,na
35,Early_K2_G8_S5,P0031,Spectral_Early,2,0.221086,8.0,5.0,na
40,Early_K2_G8_S7,P0031,Spectral_Early,2,0.221086,8.0,7.0,na



Groups with independently verified ARI=1: 53

PARTITION AUDIT COMPLETED
Saved: /content/drive/MyDrive/Research_Climate_FoodSecurity_85Kab/03_outputs/05_extended_gridsearch_R0/20261019_014117_UTC
